In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
from pyproj import Transformer
from scipy.interpolate import CubicSpline
from scipy.signal import savgol_filter

df = pd.read_csv('../data/final_track_coordinates.csv')

transformer = Transformer.from_crs("epsg:4326", "epsg:32639", always_xy=True)
df['E'], df['N'] = transformer.transform(df['longitude'].values, df['latitude'].values)

df = df.sort_index().reset_index(drop=True)
dx = np.diff(df['E'].values)
dy = np.diff(df['N'].values)
ds = np.sqrt(dx**2 + dy**2)
s = np.concatenate([[0], np.cumsum(ds)])
df['s'] = s

cs_E = CubicSpline(df['s'], df['E'])
cs_N = CubicSpline(df['s'], df['N'])

dx_ds = cs_E.derivative(1)(df['s'])
dy_ds = cs_N.derivative(1)(df['s'])

d2x_ds2 = cs_E.derivative(2)(df['s'])
d2y_ds2 = cs_N.derivative(2)(df['s'])

kappa_c = (dx_ds * d2y_ds2 - dy_ds * d2x_ds2) / (dx_ds**2 + dy_ds**2)**1.5
kappa_c = savgol_filter(kappa_c, window_length=21, polyorder=4, mode='interp')

norm = np.sqrt(dx_ds**2 + dy_ds**2)
nx = -dy_ds / norm
ny = dx_ds / norm

track_df = pd.DataFrame({
    's': df['s'],
    'E': df['E'],
    'N': df['N'],
    'kappa_c': kappa_c,
    'nx': nx,
    'ny': ny
})

track_df.to_csv('../data/track_centerline.csv', index=False)
print("Track data exported to track_centerline.csv")

fig = px.scatter(
    track_df, 
    x='E', 
    y='N',
    color='kappa_c',
    color_continuous_scale='bluered',
    labels={'kappa_c': 'Curvature'},
    title='Track Colored by Curvature'
)

fig.update_layout(
    xaxis_title='Easting', 
    yaxis_title='Northing',
    xaxis=dict(scaleanchor="y", scaleratio=1), 
    template="plotly_dark"
)

fig.show()

Track data exported to track_centerline.csv
